# GLM and Permutation Tests

In [63]:
from statsmodels.stats.multitest import multipletests

from glm_permutation_tests import run_glm, plot_glm_coefficients
# Notebook header
%load_ext autoreload
%autoreload 2
import pandas as pd
from analyses.spike_count import prepare_binned_spike_data, aggregate_trial_level
import warnings
from scipy.stats import ConstantInputWarning

warnings.simplefilter("ignore", ConstantInputWarning)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load Session for Analysis

In [64]:
# Step 1: Load and prepare data
date = "2023-09-26"
round_no = 3
bin_size = 0.05

analysis_df = prepare_binned_spike_data(date, round_no, bin_size)


Reading Intan Technologies RHD2000 Data File, Version 3.2

Found 24 amplifier channels.
Found 0 auxiliary input channels.
Found 0 supply voltage channels.
Found 0 board ADC channels.
Found 2 board digital input channels.
Found 0 board digital output channels.
Found 0 temperature sensors channels.

Header file contains no data.  Amplifiers were sampled at 20.00 kS/s.
Done!  Elapsed time: 0.0 seconds


## Effect of Stimulus Identity for Zombies

In [65]:
zombies_df = analysis_df[analysis_df['MonkeyGroup'] == 'Zombies']
formula = "SpikeCount ~ C(MonkeyName)"  # Stimulus identity


## GLM

In [73]:
glm_results = run_glm(zombies_df, formula=formula)
print(glm_results.head())
# ---- multiple comparison correction ---
# select p-value column from glm_results
pvals = glm_results['P>|z|']

Running GLM per neuron: 100%|██████████| 32/32 [00:00<00:00, 64.71it/s]

                   index      Coef.      Std.Err.             z     P>|z|  \
0              Intercept -28.128402  38133.918272 -7.376216e-04  0.999411   
1  C(MonkeyName)[T.143H]  23.364094  38133.918275  6.126854e-04  0.999511   
2  C(MonkeyName)[T.151J]  -0.000007  55501.364509 -1.290195e-10  1.000000   
3   C(MonkeyName)[T.67G]  -0.000007  53962.051818 -1.327011e-10  1.000000   
4   C(MonkeyName)[T.69X]  23.927983  38133.918274  6.274724e-04  0.999499   

          [0.025         0.975]                           NeuronID  
0  -74769.234805   74712.978001  2023-09-26_3_Channel.C_012_Unit 1  
1  -74717.742316   74764.470503  2023-09-26_3_Channel.C_012_Unit 1  
2 -108780.675539  108780.675524  2023-09-26_3_Channel.C_012_Unit 1  
3 -105763.678102  105763.678088  2023-09-26_3_Channel.C_012_Unit 1  
4  -74717.178424   74765.034389  2023-09-26_3_Channel.C_012_Unit 1  


## GLM Multiple Comparison Correction

In [74]:
# ---- multiple comparison correction ---
# select p-value column from glm_results
pvals = glm_results['P>|z|']
# correction method: 'fdr_bh' (False Discovery Rate, Benjamini/Hochberg)
reject, pvals_corrected, _, _ = multipletests(pvals, method='fdr_bh')

# add corrected p-value and reject to glm_results
glm_results['pval_corrected'] = pvals_corrected
glm_results['significant'] = reject

In [75]:
glm_results.to_excel('glm_results.xlsx')

In [76]:
plot_glm_coefficients(glm_results)

## Permutation ANOVA

In [71]:
zombies_trial_df= aggregate_trial_level(zombies_df)
unique_neurons = zombies_trial_df['NeuronID'].unique()
neuron_df = zombies_trial_df[zombies_trial_df['NeuronID'] == unique_neurons[0]]
neuron_df

,NeuronID,TaskField,MonkeyName,MonkeyGroup,SpikeCount
0,2023-09-26_3_Channel.C_002_Unit 1,1695753699241000,143H,Zombies,4
1,2023-09-26_3_Channel.C_002_Unit 1,1695753699414000,143H,Zombies,1
2,2023-09-26_3_Channel.C_002_Unit 1,1695753699562000,7124,Zombies,0
3,2023-09-26_3_Channel.C_002_Unit 1,1695753699874000,110E,Zombies,1
4,2023-09-26_3_Channel.C_002_Unit 1,1695753699967000,7124,Zombies,7
...,...,...,...,...,...
79,2023-09-26_3_Channel.C_002_Unit 1,1695753724119000,67G,Zombies,1
80,2023-09-26_3_Channel.C_002_Unit 1,1695753724353000,94B,Zombies,2
81,2023-09-26_3_Channel.C_002_Unit 1,1695753724629000,94B,Zombies,0
82,2023-09-26_3_Channel.C_002_Unit 1,1695753724866000,151J,Zombies,0


In [77]:
from glm_permutation_tests import run_permutation_anova

perm_anova_results = run_permutation_anova(zombies_trial_df, category_col='MonkeyName')

ImportError: cannot import name 'run_permutation_anova' from 'glm_permutation_tests' (/home/connorlab/Documents/GitHub/Julie/src/analyses/glm_permutation_tests.py)